# 02 — Exploratory Data Analysis (EDA)

**Phase 1:** Sales & Profitability  
**Dataset:** Superstore (Kaggle: `vivek468/superstore-dataset-final`)  
**Database:** `../data/superstore.db`  
**SQL scripts:** `../sql/`

---
**Contents:**

1. Analysis Objective
2. Setup
3. Data Quality
4. EDA Sections
 - 4.1 Global KPIs
 - 4.2 Performance by Category
 - 4.3 Discount Impact on Profit Margin
 - 4.4 Regional Performance
 - 4.5 Annual Trends & Monthly Seasonality
 - 4.6 Customer Segments & Ship Mode
 - 4.7 Product Analysis
 - 4.8 Customer Analysis
 
5. Key Findings & Insights

---
## 1. Analysis Objective

The objective of this notebook is to perform a structured exploratory analysis of the Superstore dataset, moving from high-level business performance to more detailed patterns across key dimensions such as profitability, discounts, regions, time, products, and customers.

The purpose is to identify relevant trends, anomalies, and potential performance drivers that can guide deeper diagnostic analysis in the next phase of the project.

---
## 2. Setup

In [ ]:
import sqlite3
import re
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

pd.set_option('display.float_format', '{:,.1f}'.format)
pd.set_option('display.max_colwidth', 60)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

In [ ]:
from pathlib import Path

ROOT = Path().resolve().parent
DB_PATH = ROOT / "data" / "superstore.db"
SQL_PATH = ROOT / "sql"
VISUALS_PATH = ROOT / "visuals"

VISUALS_PATH.mkdir(exist_ok=True)

conn = sqlite3.connect(DB_PATH)

def run_sql_file(filepath):
    """Read a .sql file, strip comments, and return a list of DataFrames — one per statement."""
    with open(filepath) as f:
        content = f.read()
    content = re.sub(r'--[^\n]*', '', content)          # strip line comments
    statements = [s.strip() for s in content.split(';') if s.strip()]
    return [pd.read_sql_query(s, conn) for s in statements]

def fmt_usd(val):
    return f'${val:,.0f}'

print('Connected to', DB_PATH)

---
## 3. Data Quality

Before any exploratory analysis, the data is audited for completeness and consistency.

Important: this notebook does not clean or modify the dataset. The cleaning step happens in `01_load_data.ipynb`, where the raw CSV is processed and written into the SQLite database. In this notebook, the raw CSV is inspected only for audit purposes, while all business analysis below uses the cleaned SQLite database.

The checks include:

- shape and column types
- missing values
- duplicates
- date-range sanity
- numeric ranges
- cleaned database validation

The dataset is a public Kaggle sample, so it is mostly clean. However, verifying is cheaper than assuming.

In [ ]:
df = pd.read_csv('../data/Sample - Superstore.csv', encoding='latin-1')

print(f'Shape (raw CSV): {df.shape[0]:,} rows x {df.shape[1]} columns')

print('\nColumn types:')
display(df.dtypes.to_frame('dtype'))

print('\nFirst 5 rows:')
display(df.head())

print('\nMissing values per column:')
display(df.isna().sum().to_frame('nulls'))

In [ ]:
print('Numeric summary:')
display(df.describe().T)

In [ ]:
df['_order_date'] = pd.to_datetime(df['Order Date'], format='%m/%d/%Y')
df['_ship_date']  = pd.to_datetime(df['Ship Date'],  format='%m/%d/%Y')

print(f"Order Date range: {df['_order_date'].min().date()} -> {df['_order_date'].max().date()}")
print(f"Ship Date  range: {df['_ship_date'].min().date()} -> {df['_ship_date'].max().date()}")

violations = (df['_ship_date'] < df['_order_date']).sum()
print(f'\nShip Date earlier than Order Date: {violations} row(s)')

### Duplicate Audit

The raw CSV is checked for duplicates at two levels:

1. **Fully duplicated rows:** exact matches across all columns, including `Row ID`.
2. **Duplicated business records:** rows that match across all columns except `Row ID`.

Because `Row ID` is a unique technical identifier, a duplicated business record may not appear as a fully duplicated row if only the `Row ID` differs.

In [ ]:
print(f'Fully duplicated rows in raw CSV: {df.duplicated().sum()}')
print(f'Row ID is unique in raw CSV: {df["Row ID"].is_unique}')

duplicate_check_cols = [
    'Order ID',
    'Order Date',
    'Ship Date',
    'Ship Mode',
    'Customer ID',
    'Customer Name',
    'Segment',
    'Country',
    'City',
    'State',
    'Postal Code',
    'Region',
    'Product ID',
    'Category',
    'Sub-Category',
    'Product Name',
    'Sales',
    'Quantity',
    'Discount',
    'Profit'
]

duplicate_rows_excluding_row_id = df[
    df.duplicated(subset=duplicate_check_cols, keep=False)
]

print(
    f'\nDuplicated rows excluding Row ID in raw CSV: '
    f'{len(duplicate_rows_excluding_row_id)}'
)

display(
    duplicate_rows_excluding_row_id[
        ['Row ID'] + duplicate_check_cols
    ].sort_values(['Order ID', 'Product ID', 'Row ID'])
)

### Cleaned Database Validation

The cleaned SQLite database is validated to confirm that the cleaning step from `01_load_data.ipynb` was applied correctly and that the database is ready for exploratory analysis.

In [ ]:
db_check = pd.read_sql_query(
    '''
    SELECT 
        "Row ID",
        "Order ID",
        "Product ID",
        Quantity,
        Sales,
        Discount,
        Profit
    FROM orders
    WHERE "Order ID" = 'US-2014-150119'
      AND "Product ID" = 'FUR-CH-10002965'
    ''',
    conn,
)

print('Rows remaining in cleaned SQLite database for the identified duplicate:')
display(db_check)

db_summary = pd.read_sql_query(
    '''
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT "Row ID") AS unique_row_ids,
        SUM(CASE WHEN "Row ID" = 3407 THEN 1 ELSE 0 END) AS removed_duplicate_present,
        MIN("Order Date") AS min_order_date,
        MAX("Order Date") AS max_order_date,
        SUM(CASE WHEN Sales IS NULL THEN 1 ELSE 0 END) AS null_sales,
        SUM(CASE WHEN Profit IS NULL THEN 1 ELSE 0 END) AS null_profit,
        SUM(CASE WHEN Discount IS NULL THEN 1 ELSE 0 END) AS null_discount
    FROM orders
    ''',
    conn,
)

display(db_summary)

db_repeated_pairs = pd.read_sql_query(
    '''
    SELECT 
        "Order ID",
        "Product ID",
        COUNT(*) AS line_count
    FROM orders
    GROUP BY 
        "Order ID",
        "Product ID"
    HAVING COUNT(*) > 1
    ORDER BY line_count DESC
    ''',
    conn,
)

print(f'Repeated (Order ID, Product ID) pairs in cleaned DB: {len(db_repeated_pairs)}')
display(db_repeated_pairs)

### Data Quality Summary

| Check | Result |
|---|---|
| Raw CSV shape | 9,994 rows x 21 columns |
| Missing values in raw CSV | 0 across all columns |
| Order Date range | 2014-01-03 -> 2017-12-30 |
| Ship Date earlier than Order Date | 0 violations |
| Row ID uniqueness in raw CSV | Unique |
| Fully duplicated rows in raw CSV | 0 |
| Duplicated rows excluding `Row ID` in raw CSV | 2 rows / 1 duplicated record |
| Cleaning action in `01_load_data.ipynb` | Row ID 3407 removed before writing to SQLite |
| Cleaned SQLite database shape | 9,993 rows |
| Removed duplicate present in cleaned database | No |
| Duplicated rows excluding `Row ID` in cleaned database | 0 |
| Nulls in key numeric fields in cleaned database | 0 in Sales, Profit, and Discount |

- **Raw CSV validation:** The raw CSV contains no missing values and no invalid shipping dates.

- **Duplicate found:** The raw CSV contains no fully duplicated rows when all 21 columns are considered. However, when excluding `Row ID`, the audit identifies **2 duplicated rows**, representing **1 duplicated business record**. Row IDs **3406** and **3407** contain the same values across all business-relevant fields.

- **Action taken:** The cleaning step is handled in `01_load_data.ipynb`, where Row ID **3407** is removed before the data is written to SQLite. This notebook does not modify the dataset; it verifies that the cleaned database was created correctly.

- **Database validation:** The cleaned SQLite database contains **9,993 rows**, confirms that Row ID **3407** is no longer present, has **0 duplicated rows excluding `Row ID`**, and has no null values in key numeric fields. Therefore, the database is considered ready for exploratory analysis.

---
## 4. Exploratory DATA Analysis (EDA)

---
### 4.1 Global KPIs

In [ ]:
results = run_sql_file(SQL_PATH / "01_global_metrics.sql")

kpis = results[0].T
kpis.columns = ['Value']
kpis

#### Key Observations:

- The business generated **$2.3M in sales** and **$286K in profit** across 5,009 orders and 793 customers — an overall **12.5% margin**.
- **18.7% of line items are individually unprofitable**, meaning nearly 1 in 5 transactions generates a loss at the line level.
- **52% of orders carry a discount**, indicating widespread discounting and the most likely driver of the high loss-transaction rate.
- Average revenue per order is ~$458; average orders per customer is ~6.3, suggesting moderate repeat purchasing across the base.

---
### 4.2 Performance by Category

In [ ]:
results = run_sql_file(SQL_PATH / "02_by_category.sql")
cat_df  = results[0]
sub_df  = results[1]

print('=== Category ===')
display(cat_df)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(cat_df['Category'], cat_df['sales'], color='steelblue')
axes[0].set_title('Sales by Category')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e6:.1f}M'))

axes[1].barh(cat_df['Category'], cat_df['margin_pct'],
             color=['#d62728' if v < 5 else 'steelblue' for v in cat_df['margin_pct']])
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Profit Margin % by Category')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig(VISUALS_PATH / "02a_category_performance.png", bbox_inches="tight")
plt.show()

In [ ]:
print('=== Sub-Category (sorted by profit) ===')
display(sub_df)

fig, ax = plt.subplots(figsize=(10, 8))
colors = ['#d62728' if v < 0 else 'steelblue' for v in sub_df['profit']]
ax.barh(sub_df['Sub-Category'], sub_df['profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Profit by Sub-Category')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig(VISUALS_PATH / "02b_subcategory_profit.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- The three categories are nearly equal in revenue share (~31–36%), but margin differs dramatically: **Technology (17.4%) and Office Supplies (17.0%) are healthy; Furniture (2.5%) is not**.
- Furniture accounts for **32% of revenue but only 6.4% of total profit** — a structural drag on overall margin that is not offset by its volume.
- Within Furniture, **Tables (–$17.7K) and Bookcases (–$3.5K)** have never generated positive cumulative profit over 4 years. Supplies (–$1.2K) also runs at a loss.
- Technology is the most valuable category: highest margin and highest absolute profit ($145K of the $286K total).

---
### 4.3 Discount Impact on Profit Margin

> **Breakeven point ~20% discount.** Every discount ≥ 20% produces negative average margin.

In [ ]:
results  = run_sql_file(SQL_PATH / '03_discount_impact.sql')
disc_df  = results[0]
display(disc_df)

fig, ax1 = plt.subplots(figsize=(10, 5))

bar_colors = ['#d62728' if v < 0 else 'steelblue' for v in disc_df['margin_pct']]
bars = ax1.bar(disc_df['discount_bucket'], disc_df['items'], color='lightsteelblue', label='Items')
ax1.set_ylabel('Items')
ax1.set_xlabel('Discount Bucket')

ax2 = ax1.twinx()
ax2.plot(disc_df['discount_bucket'], disc_df['margin_pct'], color='crimson',
         marker='o', linewidth=2, label='Margin %')
ax2.axhline(0, color='gray', linewidth=0.8, linestyle='--')
ax2.set_ylabel('Profit Margin %')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper right')

plt.title('Discount Bucket: Item Count vs Profit Margin %')
plt.tight_layout()
plt.savefig(VISUALS_PATH / "03_discount_impact.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- The margin decline is steep and consistent: **29.5% at 0% → 11.6% at 11–20% → –10.1% at 21–30% → –24.8% at 31–50%**. The 20% mark is a hard breakeven threshold.
- The discount distribution is bimodal: most items are either undiscounted (4,798) or in the 11–20% range (3,709). Only 94 items fall in the 1–10% range, suggesting discounting is applied as a significant lever rather than minor adjustments.
- The **>50% bucket (856 items)** is larger than the 21–50% range combined and averages –119.2% margin — the company loses more in profit than it collects in revenue on those transactions.

---
### 4.4 Regional Performance

In [ ]:
results    = run_sql_file(SQL_PATH / '04_by_region.sql')
region_df  = results[0]
states_df  = results[1]

print('=== Region ===')
display(region_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(region_df['Region'], region_df['sales'], color='steelblue')
axes[0].set_title('Sales by Region')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[1].barh(region_df['Region'], region_df['margin_pct'], color='steelblue')
axes[1].set_title('Profit Margin % by Region')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig(VISUALS_PATH / "04a_region_performance.png", bbox_inches="tight")
plt.show()

In [ ]:
print('=== States with Highest Losses ===')
display(states_df)

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if v < 0 else 'steelblue' for v in states_df['profit']]
ax.barh(states_df['State'], states_df['profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Profit by State (Bottom 10)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig(VISUALS_PATH / "04b_state_losses.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- **West leads in both revenue ($725K) and margin (14.9%)**. Central has comparable revenue ($501K) but nearly half the margin (7.9%) — the largest performance gap in the dataset.
- The South is more efficient than its revenue rank implies: $392K in sales at 11.9% margin, generating better returns per dollar sold than Central despite lower volume.
- **Texas alone accounts for –$25.7K in losses**, followed by Ohio (–$17.0K) and Pennsylvania (–$15.6K). These three states together represent over $58K in losses — a material drag on overall margin.

---
### 4.5 Annual Trends & Monthly Seasonality

In [ ]:
results    = run_sql_file(SQL_PATH / '05_trends.sql')
annual_df  = results[0]
monthly_df = results[1]

print('=== Annual Trend ===')
display(annual_df)

fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.bar(annual_df['year'], annual_df['sales'], color='lightsteelblue', label='Sales')
ax1.set_ylabel('Sales ($)')
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

ax2 = ax1.twinx()
ax2.plot(annual_df['year'], annual_df['margin_pct'], color='crimson',
         marker='o', linewidth=2, label='Margin %')
ax2.set_ylabel('Margin %')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Annual Sales & Profit Margin (2014–2017)')
plt.tight_layout()
plt.savefig(VISUALS_PATH / "05a_annual_trend.png", bbox_inches="tight")
plt.show()

In [ ]:
month_labels = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
monthly_df['month_name'] = monthly_df['month'].apply(lambda m: month_labels[m - 1])

print('=== Monthly Seasonality ===')
display(monthly_df[['month_name','items','sales','profit','margin_pct']])

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(monthly_df['month_name'], monthly_df['sales'], color='steelblue')
ax.set_title('Monthly Sales (all years combined)')
ax.set_ylabel('Sales ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig(VISUALS_PATH / "05b_monthly_seasonality.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- Revenue grew from **$484K (2014) to $733K (2017)** — a 51% increase — but margin peaked at **13.4% in 2016** and softened to **12.7% in 2017**. Revenue growth is not fully translating into profit growth.
- 2015 is the only year with lower sales than the prior year ($471K vs $484K), yet margin improved to 13.1%, suggesting volume was traded for quality that year.
- **November ($352K) and December ($325K)** are the top two revenue months, with September ($308K) third. The Q4 spike is the clearest and most actionable seasonality pattern in the dataset.

---
### 4.6 Customer Segments & Ship Mode

In [ ]:
results   = run_sql_file(SQL_PATH / '06_by_segment.sql')
seg_df    = results[0]
ship_df   = results[1]

print('=== Segment ===')
display(seg_df)
print('\n=== Ship Mode ===')
display(ship_df)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].barh(seg_df['Segment'], seg_df['margin_pct'], color='steelblue')
axes[0].set_title('Profit Margin % by Segment')
axes[0].set_xlabel('%')

axes[1].barh(ship_df['Ship Mode'], ship_df['margin_pct'], color='steelblue')
axes[1].set_title('Profit Margin % by Ship Mode')
axes[1].set_xlabel('%')

plt.tight_layout()
plt.savefig(VISUALS_PATH / "06_segment_shipmode.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- **Consumer is the largest segment by revenue ($1.16M) but has the lowest margin (11.5%)**. Home Office is the smallest ($429K) yet the healthiest (14.0%), suggesting larger segments face greater discount pressure.
- For Ship Mode, **First Class delivers the highest margin (13.9%)** despite its premium positioning. Standard Class handles the most volume ($1.36M) at the lowest margin (12.1%).
- The margin spread across ship modes is narrow (12.1%–13.9%), unlike the sharp gaps seen in discount buckets or across categories. Ship mode is not a primary margin lever.

---
### 4.7 Product Analysis

In [ ]:
results      = run_sql_file(SQL_PATH / '07_products.sql')
top_prod_df  = results[0]
loss_prod_df = results[1]
count_loss   = results[2]

print('=== Top 10 Products by Profit ===')
display(top_prod_df)

print('\n=== Top 10 Products with Highest Losses ===')
display(loss_prod_df)

print(f"\nProducts with negative cumulative profit: {count_loss['products_at_loss'].iloc[0]}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(top_prod_df['Product Name'].str[:40], top_prod_df['profit'], color='steelblue')
axes[0].set_title('Top 10 Products by Profit')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

axes[1].barh(loss_prod_df['Product Name'].str[:40], loss_prod_df['profit'], color='#d62728')
axes[1].axvline(0, color='black', linewidth=0.8)
axes[1].set_title('Top 10 Products — Highest Losses')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))

plt.tight_layout()
plt.savefig(VISUALS_PATH / "07_product_analysis.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- The **Canon imageCLASS 2200 Copier (+$25.2K)** is the single largest profit contributor, representing ~8.8% of total profit on its own. The top three earners are all copier or laser printer devices.
- The three worst loss-makers — **Cubify CubeX Double Head (–$8.9K), Lexmark MX611dhe (–$4.6K), and Cubify CubeX Triple Head (–$3.8K)** — are concentrated in the Machines sub-category, with two being variants of the same product line.
- **302 SKUs have generated cumulative losses** over the full 4-year period, representing a persistent portfolio drag that is unlikely to self-correct without active review.

---
### 4.8 Customer Analysis

In [ ]:
results    = run_sql_file(SQL_PATH / '08_customers.sql')
top_cust   = results[0]
cust_dist  = results[1]

print('=== Top 20 Customers by Sales ===')
display(top_cust)

print('\n=== Customer Profit Distribution ===')
display(cust_dist)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#d62728' if v < 0 else 'steelblue' for v in top_cust['profit']]
ax.barh(top_cust['Customer Name'], top_cust['profit'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Top 20 Customers by Sales — Profit ($)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e3:.0f}K'))
plt.tight_layout()
plt.savefig(VISUALS_PATH / "08_customer_profit.png", bbox_inches="tight")
plt.show()

#### Key Observations:

- **Sean Miller is the top spender ($25K in sales) but generates a net loss of –$1,981** — the clearest example of a customer who drives revenue while destroying margin.
- **155 customers (19.5% of the 793-customer base) are net loss-generating** over the full period. Nearly 1 in 5 customers costs the business money in aggregate.
- Profit is heavily concentrated: only **25 customers (3.2%) exceed $2K in cumulative profit**, while 417 (52.6%) fall in the $0–$500 range. The top accounts carry a disproportionate share of total earnings.

---
## 5. Key Findings 

This section summarizes the main patterns identified during the exploratory analysis. These findings are not final recommendations; they are diagnostic leads that should guide deeper analysis in the next phase.

### 1. Discounts above 20% are associated with negative margins

- Items sold at zero discount run at **29.5% margin**. Once discounts reach 21–30%, margin turns negative (–10.1%); the 31–50% bucket averages –24.8%.
- The **>50% discount bucket (856 items) averages –119.2% margin** — the company loses more in direct profit than it collects in revenue on those transactions.
- *→ Section 4.3: Discount Impact chart.*


### 2. Furniture has a structural profitability problem
- Furniture generates **$742K in revenue (32% of total) but only $18K in profit (2.5% margin)** — roughly seven times below Technology (17.4%) and Office Supplies (17.0%). 
- Two sub-categories are outright loss-makers in aggregate: **Tables (–$17.7K)** and **Bookcases (–$3.5K)**. 
- *→ Section 2: Sub-Category Profit chart.*


### 3. Central region underperforms and Texas is the single biggest loss center
- Central delivers only **7.9% margin** compared to West's **14.9%** on a similar order base. 
- At the state level, **Texas alone accounts for –$25.7K in losses** — more than any other state. Ohio (–$17K) and Pennsylvania (–$15.6K) follow. 
- *→ Section 4: Region Performance and State Losses charts.*


### 4. Revenue grew +51% from 2014–2017, but margin slipped in 2017
- Annual sales climbed from **$484K (2014) to $733K (2017)** — a 51% increase over four years. 
- Profit margin peaked at **13.4% in 2016** and declined to **12.7% in 2017**, despite higher volume. 
- Whether this reflects a category mix shift toward lower-margin products or deeper discounting is an open question. 
- *→ Section 5: Annual Trend chart.*


### 5. Q4 (Oct–Dec) is the peak season — plan around it
- November and December are consistently the two highest-revenue months across all four years. 
- Sales and profit spike sharply in Q4 relative to the rest of the year. 
- *→ Section 5: Monthly Seasonality chart.*


### 6. 302 catalog products carry cumulative losses — review for discontinuation
- Of all SKUs sold over the four-year period, **302 have lost money in aggregate**. 
- The worst individual products (led by Cubify CubeX 3D printers) have destroyed thousands of dollars in value. 
    - These are not temporarily underperforming — they are structurally loss-making at current pricing and discount levels. 
- *→ Section 7: Product Analysis chart.*


### 7. The highest-spending customers are not always the most profitable
- Two of the top-20 customers ranked by total sales volume show **negative total profit** over the period. 
- Revenue concentration is not the same as profitability concentration. 
- *→ Section 8: Customer Profit chart.*


### EDA Findings Summary

| Finding | Evidence | Phase 2 Question |
|---|---|---|
| Discount Breakeven | ≥20% discount → negative average margin | Which products, regions, and customers drive high-discount losses? |
| Furniture Problem | 32% of sales, only 6.4% of profit (2.5% margin) | Is the issue driven by discounting, supplier costs, or product mix? |
| Central (Worst Region) | Central: 7.9% margin vs West 14.9% | Which states, products, or customers explain the gap? |
| Texas (Worst State) | Texas shows -$25.7K profit | Are losses concentrated in specific sub-categories or accounts? |
| 2017 margin dip | +51% over 4 years, but margin dipped in 2017 | Was the decline caused by mix shift, discounting, or pricing? |
| Peak Season | Q4 (Oct–Dec): highest sales and profit | Are Q4 promotions margin-accretive or margin-destructive? |
| Product value destroyers | Tables (-$17.7K), Bookcases (-$3.5K), Cubify printers | Are losses structural or caused by isolated discounting? |
| Customer profitability gap | Some top spenders are unprofitable | Which customers generate revenue without profit? |

**Next:** Phase 2 — business interpretation and actionable recommendations.

In [ ]:
conn.close()
print('Done. Visuals saved to ../visuals/')